In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA SILVER
gold_silver = spark.table("spotify_catalog.silver.spotify_tracks")

display(gold_silver.limit(10))

In [0]:
display(gold_silver)

**DIMENSIONES:**

- DIM_ARTIST: pk_artist, artist_id, artist_name
- DIM_ALBUM : pk_album, album_id, album_name, album_type, release_date, release_date_precision, release_date_clean, total_tracks, artist_id
- DIM_TRACK : pk_track, track_id, track_name, duration_seconds, explicit, popularity, popularity_level, album_id
- DIM_DATE  : pk_date, full_date, year, quarter, month, month_name, week, day_of_week, day_name




**DIM_ARTIST**

In [0]:
dim_artist_base = (
    gold_silver
    .select(
        "artist_id",
        "artist_name",
        "artist_href",
        "artist_uri"
    )
    .filter(col("artist_id").isNotNull())
    .dropDuplicates(["artist_id"])
)

In [0]:
window_artist = Window.orderBy("artist_id")

dim_artist = (
    dim_artist_base
    .withColumn(
        "sk_artist",
        row_number().over(window_artist)
    )
    .select(
        "sk_artist",
        "artist_id",
        "artist_name",
        "artist_href",
        "artist_uri"
    )
)

display(dim_artist)

In [0]:
(
    dim_artist
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_artist"
    )
)

**DIM_ALBUM**

In [0]:
dim_album_base  = (
    gold_silver
    .select(
        "album_id",
        "album_name",
        "album_type",
        "release_date",
        "release_date_precision",
        "release_date_clean",
        "album_total_tracks",
        "artist_id"
    )
    .filter(col("album_id").isNotNull())
    .dropDuplicates(["album_id"])
)


In [0]:
window_album = Window.orderBy("album_id")

dim_album = (
    dim_album_base
    .withColumn(
        "sk_album",
        row_number().over(window_album)
    )
    .select(
        "sk_album",
        "album_id",
        "album_name",
        "album_type",
        "release_date",
        "release_date_precision",
        "release_date_clean",
        "album_total_tracks"
    )
)

display(dim_album)

In [0]:
(
    dim_album
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_album"
    )
)

**DIM_TRACK**

In [0]:
dim_track_base = (
    gold_silver
    .select(
        "track_id",
        "track_name",
        "duration_seconds",
        "explicit",
        "popularity",
        "popularity_level",
        "album_id"
    )
    .filter(col("track_id").isNotNull())
    .dropDuplicates(["track_id"])
)


In [0]:
window_track = Window.orderBy("track_id")

dim_track = (
    dim_track_base
    .withColumn(
        "sk_track",
        row_number().over(window_track)
    )
    .select(
        "sk_track",
        "track_id",
        "track_name",
        "duration_seconds",
        "explicit",
        "popularity",
        "popularity_level",
        "album_id"
    )
)

display(dim_track)

In [0]:
(
    dim_track
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_track"
    )
)

**DIM_DATE**

In [0]:
date_min = gold_silver.select(min("release_date_clean")).first()[0]
date_max = gold_silver.select(max("release_date_clean")).first()[0]

print("Fecha mínima:", date_min)
print("Fecha máxima:", date_max)

In [0]:
dim_date = spark.sql(f"""
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS sk_date,
    d AS full_date,
    year(d) AS year,
    quarter(d) AS quarter,
    month(d) AS month,
    date_format(d, 'MMMM') AS month_name,
    weekofyear(d) AS week_of_year,
    dayofmonth(d) AS day_of_month,
    dayofweek(d) AS day_of_week,
    date_format(d, 'EEEE') AS day_name
FROM (
    SELECT explode(
        sequence(
            to_date('{date_min}'),
            to_date('{date_max}'),
            interval 1 day
        )
    ) AS d
)
""")

display(dim_date)

In [0]:
(
    dim_date
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_date"
    )
)

**FACT_TRACK**

In [0]:
fact_track = (
    gold_silver.alias("s")

    .join(
        dim_track.alias("t"),
        col("s.track_id") == col("t.track_id"),
        "left"
    )

    .join(
        dim_artist.alias("a"),
        col("s.artist_id") == col("a.artist_id"),
        "left"
    )

    .join(
        dim_album.alias("al"),
        col("s.album_id") == col("al.album_id"),
        "left"
    )

    .join(
        dim_date.alias("d"),
        col("s.release_date_clean") == col("d.full_date"),
        "left"
    )
)

display(fact_track.limit(10))

In [0]:
fact_track = fact_track.select(
    
    # Claves dimensionales
    col("t.sk_track"),
    col("a.sk_artist"),
    col("al.sk_album"),
    col("d.sk_date"),

    # Claves naturales
    col("s.track_id"),
    col("s.artist_id"),
    col("s.album_id"),

    # Métricas
    col("s.popularity"),
    col("s.popularity_level"),
    col("s.duration_seconds"),

    # Indicadores
    col("s.explicit"),

    # Información temporal
    col("s.release_date"),
    col("s.release_date_precision"),   
    col("s.release_date_clean"),

    # Información de extracción
    col("s.extraction_timestamp"),
    col("s.search_term")
)

display(fact_track)

In [0]:
(
    fact_track
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.fact_track"
    )
)